# 🌱 Green Self-Healing C++ Compiler: Final Demo
**One tool, one pipeline, needs only g++:** compile → explain → classify → security-check → auto-repair.

**How to use this notebook**
1. Run **Section 0** first (it asks for `cd1_final.zip` if it isn't uploaded yet).
2. Then use **Runtime ▸ Run all**, or press ▶ on one section at a time while presenting.
3. Each section has a **SAY** line (what to tell the professor) and a **SHOW** line (what to point at).
4. If anything breaks: *Runtime ▸ Restart session*, then run Section 0 again.

Total run time ≈ 6–8 minutes. Run it **once before the presentation** and save (File ▸ Save) so results stay on screen.

## 0. Setup (run first)

In [ ]:
import glob, os, subprocess
if not glob.glob('/content/cd1_final*.zip'):
    from google.colab import files
    print("Choose cd1_final.zip ...")
    files.upload()
zips = sorted(glob.glob('/content/cd1_final*.zip'), key=os.path.getmtime)
subprocess.run("apt-get install -y -qq g++ binutils > /dev/null 2>&1", shell=True)
subprocess.run(f"rm -rf /content/project && mkdir -p /content/project && unzip -q -o '{zips[-1]}' -d /content/project", shell=True)
root = os.path.dirname(os.path.dirname(glob.glob('/content/project/**/tests/test_optimized.py', recursive=True)[0]))
os.chdir(root)
subprocess.run("pip install -q -r requirements.txt", shell=True)
os.environ.update(GSC_DISABLE_DATASET_LOGGING="1", QT_QPA_PLATFORM="offscreen", PYTHONWARNINGS="ignore")
print("Project folder:", root)
print(subprocess.run("g++ --version | head -1", shell=True, capture_output=True, text=True).stdout)

## 1. Evidence first: every test passes
**SAY:** *"48 automated tests, all passing on Linux with a real g++. On a machine without g++ they are reported as SKIPPED, never silently passed."*
**SHOW:** `32/32 passed` and `16/16 passed`.

In [ ]:
%%bash
python3 tests/test_optimized.py 2>&1 | tail -3
python3 tests/test_audit_fixes.py 2>&1 | tail -1

## 2. Detection: a bug the original tool called "clean"
**SAY:** *"A null-pointer dereference. The original version said 'Compilation successful'. Now it reports the category, CWE-476 and a plain-English explanation."*
**SHOW:** the Category line (CWE-476, 100% via rule) and the What / Why / How-to-fix block.

In [ ]:
%%bash
python3 src/main.py demo/01_null_deref.cpp | head -22

## 3. Security analysis with industry CWE ids
**SAY:** *"It follows untrusted input into dangerous calls and labels every finding with a CWE number, severity and confidence level."*
**SHOW:** the CRITICAL use-after-free, the HIGH format string, mismatched `delete`, and `(medium confidence)` on strcpy.

In [ ]:
%%bash
python3 src/main.py demo/02_vulnerable.cpp --no-optimize | sed -n '/critical,/,$p'

## 4. Verified auto-hardening
**SAY:** *"A security fix is kept only if the program still compiles AND the finding disappears. It never edits your file unless you ask."*
**SHOW:** `4 verified fix(es)`, the diff, and the one finding it honestly **could not** fix (unvalidated array index).

In [ ]:
%%bash
cp demo/02_vulnerable.cpp /tmp/vuln.cpp
python3 src/main.py /tmp/vuln.cpp --harden | head -45

## 5. The original purpose: explain compile errors
**SAY:** *"A normal syntax error, explained in plain English instead of cryptic compiler text."*

In [ ]:
%%bash
python3 src/main.py demo/04_syntax_error.cpp | head -24

## 6. ⭐ The self-healing loop
**SAY:** *"The core idea: compile → classify → apply ONE rule-based fix → recompile, repeat until clean or 10 attempts. The compiler itself is the verifier. The classifier uses ML, but the fixes are deterministic rules, so there are no made-up patches and no LLM involved."*
**SHOW:** attempt 1 and 2 each fix one missing `;`, attempt 3 recompiles clean. g++ reports the error on the line *after* the real mistake and the fixer accounts for that.

In [ ]:
%%bash
cat demo/06_heal_semicolons.cpp
echo "=============== healing ==============="
python3 tools/heal_demo.py demo/06_heal_semicolons.cpp

## 7. Performance advice
**SAY:** *"It spots exponential recursion and hidden O(n²) loops, like strlen() in a loop condition."*

In [ ]:
%%bash
python3 src/main.py demo/03_slow.cpp | sed -n '/Optimization advice/,$p'

## 8. Speed: cold vs cached
**SAY:** *"The original took about 2.9 seconds per file. A cold build is now under a second, and a cached one is essentially instant."*
(Exact numbers depend on Colab's CPU. Quote the pattern, not the digits.)

In [ ]:
%%bash
rm -rf ~/.cache/gsc-build-cache
echo "cold  :"; python3 src/main.py demo/03_slow.cpp -v --no-optimize | grep timing
echo "cached:"; python3 src/main.py demo/03_slow.cpp -v --no-optimize | grep timing

## 9. Safety gate
**SAY:** *"Dangerous code such as system() is blocked before any compiler runs, so analysing untrusted code can't execute it."*

In [ ]:
%%bash
python3 src/main.py demo/05_blocked.cpp | head -12

## 10. CI integration: SARIF + exit code
**SAY:** *"Exit code 2 means 'fail the build'. SARIF is the standard format GitHub code-scanning understands."*

In [ ]:
%%bash
python3 src/main.py demo/02_vulnerable.cpp --fail-on high --format sarif 2>/dev/null | head -c 350; echo; echo ...
python3 src/main.py demo/02_vulnerable.cpp --fail-on high > /dev/null; echo "exit code = $?"

## 11. Scale test: ~8,800-line files
**SAY:** *"It handles large files. I generated two files of about 8,800 lines: one with syntax and lexical errors, one with security bugs."*
**SHOW:** errors found at line ~8,810 with exact locations (missing `;`, unterminated string, stray `@`). Lexical errors come from g++'s lexer and are labelled as syntax errors.

In [ ]:
%%bash
python3 tools/make_big_tests.py /content/big
echo "=== errors file ==="
python3 src/main.py /content/big/big_errors.cpp --no-optimize | grep -E "issue\(s\)|^ERROR|^WARNING|Location"

In [ ]:
%%bash
echo "=== security file ==="
python3 src/main.py /content/big/big_security.cpp --no-optimize | grep -E "^\[|findings|Risk score"
echo; echo "=== auto-harden the big file ==="
python3 src/main.py /content/big/big_security.cpp --no-optimize --harden | grep -E "verified|GSC-|Remaining|Written"

## 12. Real open-source code
**SAY:** *"I tested on real libraries, not just my own examples. tinyxml2 (about 3,000 lines) gives only minor warnings. lodepng (about 7,300 lines) comes back clean, which shows it doesn't invent problems."*
(A tiny `main()` is appended because the tool compiles and links; a library with no `main` would report an undefined reference.)

In [ ]:
%%bash
mkdir -p /content/real && cd /content/real
wget -q -O tinyxml2.cpp https://raw.githubusercontent.com/leethomason/tinyxml2/master/tinyxml2.cpp
wget -q -O tinyxml2.h   https://raw.githubusercontent.com/leethomason/tinyxml2/master/tinyxml2.h
wget -q -O lodepng.cpp  https://raw.githubusercontent.com/lvandeve/lodepng/master/lodepng.cpp
wget -q -O lodepng.h    https://raw.githubusercontent.com/lvandeve/lodepng/master/lodepng.h
for f in tinyxml2 lodepng; do cp $f.cpp ${f}_test.cpp; printf '\nint main(){ return 0; }\n' >> ${f}_test.cpp; done
wc -l tinyxml2_test.cpp lodepng_test.cpp

In [ ]:
%%bash
echo "=== tinyxml2 ==="; python3 src/main.py /content/real/tinyxml2_test.cpp --no-optimize | grep -E "issue\(s\)|findings|Risk score"
echo "=== lodepng  ==="; python3 src/main.py /content/real/lodepng_test.cpp --no-optimize | grep -E "issue\(s\)|Compilation|findings|Risk score"

## 13. ⭐ Plant bugs in real code, find them, fix them
**SAY:** *"I planted four classic vulnerabilities inside the real 3,000-line library. The tool found all of them with CWE ids, applied four fixes, and verified each one."*
**SHOW:** CRITICAL use-after-free, HIGH format string, HIGH mismatched delete; then `4 verified fix(es)` and `Remaining findings: 0`; then the diff of just the changed lines.

In [ ]:
%%bash
cd /content/real
cp tinyxml2.cpp injected.cpp
cat >> injected.cpp <<'EOF'

#include <cstring>
#include <cstdio>
#include <cstdlib>
void injected_copy(const char* name) { char buf[8]; strcpy(buf, name); printf("hi %s\n", buf); }
void injected_fmt(const char* msg) { printf(msg); }
void injected_free() { int* p = new int[4]; delete p; char* q = (char*)malloc(8); if (!q) return; free(q); free(q); }
int main(int argc, char** argv) { if (argc > 1) { injected_copy(argv[1]); injected_fmt(argv[1]); } injected_free(); return 0; }
EOF
wc -l injected.cpp

In [ ]:
%%bash
echo "=== FIND ==="
python3 src/main.py /content/real/injected.cpp --no-optimize | grep -E "^\[(CRITICAL|HIGH)\]|critical,"
echo; echo "=== FIX (verified) ==="
python3 src/main.py /content/real/injected.cpp --no-optimize --harden | grep -E "verified|GSC-|Remaining|Written"
echo; echo "=== CORRECTED CODE (only changed lines) ==="
diff --strip-trailing-cr /content/real/injected.cpp /content/real/injected.hardened.cpp || true

## 14. Measured results (be honest about them)
**SAY:** *"The first set was written while building the tool, so I treat it as tuned. The blind set was unseen: 17 of 20 on the first run, 19 of 20 after fixing what it exposed. I quote the blind number as the honest estimate."*

In [ ]:
%%bash
python3 benchmarks/eval_security.py
echo
sed -n 7,22p benchmarks/RESULTS.md

## 15. Limitations (say these before they are asked)
* **Not magic:** rule-based healing fixes common, well-understood mistakes. It **refuses** linker errors and access errors, and stops on a stray `@` or an undeclared variable, because those need human intent.
* **No LLM.** Classification uses a small ML model (numpy only, no GPU, no cloud, no API keys). Fixes are deterministic rules verified by g++. Future work: an optional LLM suggestion step for the cases the rules refuse, still verified by the compiler.
* **Scope:** one translation unit at a time (single file plus its local headers). A library file needs a `main()` to link.
* **Static analysis limits:** no deep inter-procedural data-flow; `medium confidence` findings need human review. The safety gate is a best-effort filter, not a sandbox.
* **Risk score:** a simple weighted count, so many harmless medium warnings can read as "High Risk". Calibrating it is future work.
* **Windows:** needs g++ (use WSL or MSYS2). Tests that need missing tools show as SKIPPED.

## 16. Likely questions
| Question | Short answer |
|---|---|
| Is it AI? | ML classifier for unusual error wording; fixes are deterministic rules. No LLM. |
| Why not cppcheck / clang-tidy? | Ours links compiler warnings, CWE labels, explanations, verified auto-fix and a repair loop in one pipeline that needs only g++. |
| Is 100% accuracy real? | The tuned set is tuned. Quote the blind set: 17/20 first run, 19/20 after fixes. |
| How do you know a fix is safe? | g++ recompiles it; for security fixes the finding must also disappear. |
| What did the real-code test teach you? | A real flaw: the hardener changed CRLF to LF line endings. I fixed it to preserve them. |